# Supply Chain Assistant with Oracle AI Agent Memory

[![Documentation](https://img.shields.io/badge/Documentation-Oracle%20AI%20Agent%20Memory-red?style=flat-square)](https://www.oracle.com/database/ai-agent-memory/)


**Framework:** Claude Agent SDK · **Memory:** Oracle AI Agent Memory on Oracle AI Database

This notebook builds a **supply chain assistant** that tracks and updates shipment cargo. The assistant exposes tools to read and mutate shipment state and uses Oracle AI Agent Memory to remember facts about shipments, customers, and operational context across sessions.

---

## Application Modes in Agent Applications

AI agent applications generally fall into three operational modes:

| Mode | Shape | Typical use |
|---|---|---|
| **Assistant** | Turn-by-turn conversational, tool-using | Operations support, customer service, internal ops copilots |
| **Deep Research** | Multi-step autonomous investigation | Literature review, scoping, market research |
| **Workflow** | Predetermined sequence with conditional branches | Approval pipelines, compliance gates, structured triage |

> **📌 This notebook focuses on the Assistant mode.**
>
> An assistant agent is conversational first — it responds to human prompts, calls tools when it needs to read or change state, and carries context across turns. Memory is how it stays coherent across a working day or across sessions with the same operator.

![Application modes](images/application_mode.png)

![The application-mode spectrum](images/application_mode_spectrum.png)


## What You'll Learn

- How to define **in-process tools** with the Claude Agent SDK using the `@tool` decorator
- How to wire those tools into an **MCP server** that the Claude Agent SDK can serve
- How to run the assistant in **multi-turn** mode with `ClaudeSDKClient`
- How to use Oracle AI Agent Memory to persist shipment records, operational notes, and conversation history so the assistant stays coherent across restarts

> **💡 Key insight:** In the Claude Agent SDK, the `@tool` decorator + `create_sdk_mcp_server` pattern is how Python functions become things the LLM can call. You don't define tools inline on the agent; you register them as an MCP server and point the agent at it.

## Prerequisites

- **Oracle AI Database** running locally (Docker container) or reachable over the network
- **`ANTHROPIC_API_KEY`** for the Claude Agent SDK
- Embeddings run locally via the open-source `nomic-embed-text-v1.5` model (fastembed / ONNX, torch-free)
- The `oracleagentmemory` wheel installed in the active kernel's environment

## 1. Install dependencies

In [1]:
# Core memory library + the Claude Agent SDK + local embeddings (no OpenAI key needed).
%pip install -q oracleagentmemory claude-agent-sdk nest_asyncio fastembed

Note: you may need to restart the kernel to use updated packages.


## 2. Configure API keys and Oracle connection

In [2]:
import os
import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your ANTHROPIC_API_KEY: ")

os.environ.setdefault("DB_USER", "VECTOR")
os.environ.setdefault("DB_PASSWORD", "VectorPwd_2025")
os.environ.setdefault("DB_CONNECT_STRING", "localhost:1521/FREEPDB1")

import nest_asyncio
nest_asyncio.apply()

print("Environment configured.")

Environment configured.


## 3. Connect to Oracle AI Database and initialise the memory client

The assistant uses memory for two things:

1. **Operational facts** — shipment ownership, customer preferences, known issues — stored as durable memories the assistant can recall by semantic search.
2. **Conversation history** — stored as messages on a thread so the assistant picks up where the operator left off.

We use the open-source `nomic-embed-text-v1.5` model (768-dim, local via fastembed) for vector search and attach Claude Haiku as the LLM so the client can automatically extract memories from thread messages.

> **💡 Key insight:** `table_name_prefix` gives each application its own namespace inside a shared Oracle schema. The supply-chain assistant's data won't collide with other OAMP-using apps in the same database.

In [3]:
import oracledb
import numpy as np
from fastembed import TextEmbedding
from oracleagentmemory.core import OracleAgentMemory
from oracleagentmemory.core.llms import Llm
from oracleagentmemory.apis.embedders.embedder import IEmbedder


def _unit(v):
    v = np.asarray(v, dtype=np.float32)
    n = np.linalg.norm(v)
    return v / n if n else v


class NomicEmbedder(IEmbedder):
    """Open-source, local embeddings: nomic-embed-text-v1.5 (768-dim) via fastembed ONNX.

    No API key and no network call — this is what lets the notebook run without OpenAI.
    fastembed applies nomic's `search_document:` / `search_query:` prefixes internally
    (passages via `embed`, queries via `query_embed`).
    """

    def __init__(self, model_name="nomic-ai/nomic-embed-text-v1.5"):
        self._model = TextEmbedding(model_name=model_name)

    def embed(self, texts, *, is_query=False):
        gen = self._model.query_embed(texts) if is_query else self._model.embed(texts)
        return np.asarray([_unit(v) for v in gen], dtype=np.float32)

    async def embed_async(self, texts, *, is_query=False):
        return self.embed(texts, is_query=is_query)


connection = oracledb.connect(
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    dsn=os.environ["DB_CONNECT_STRING"],
)
print("Connected to Oracle AI Database:", connection.version)


/Users/richmondalake/opt/anaconda3/envs/oracle_demos/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connected to Oracle AI Database: 23.26.0.0.0


In [4]:
# Local open-source embedder (no OpenAI) + Claude Haiku as the lightweight
# extraction / summary LLM (routed through litellm's `anthropic/` provider).
embedder = NomicEmbedder()
extraction_llm = Llm("anthropic/claude-sonnet-4-6", temperature=0.2)

In [ ]:
memory_client = OracleAgentMemory(
    connection=connection,
    embedder=embedder,
    llm=extraction_llm,
    extract_memories=True,
    schema_policy="create_if_necessary",
    table_name_prefix="supply_",
)
print("Memory client ready (nomic embeddings + Claude helper — no OpenAI).")

Memory client ready (nomic embeddings + Claude helper — no OpenAI).


## 4. Register the operator and the agent

Each human operator is a `user_id`, and the supply-chain assistant itself is an `agent_id`. This lets multiple operators share the same assistant with isolated memory, and lets one operator converse with multiple specialised assistants.

In [6]:
OPERATOR_ID = "operator-morgan"
AGENT_ID = "supply-chain-assistant"

for create_fn, eid, info in [
    (memory_client.add_user, OPERATOR_ID, "Operations manager — West Coast logistics desk."),
    (memory_client.add_agent, AGENT_ID, "Supply chain assistant with read/write tools for shipment state."),
]:
    try:
        create_fn(eid, info)
        print(f"Registered: {eid}")
    except ValueError:
        print(f"Already exists: {eid}")

Already exists: operator-morgan
Already exists: supply-chain-assistant


## 5. Simulate a shipment database

In production, shipment state lives in an ERP or logistics system. For this example we use an in-memory dict. The tools we expose to the agent don't care — they just need read/write methods over *some* shipment store.

> **📌 Production pattern.** Replace `SHIPMENT_DB` with calls to your real system of record (Oracle tables, an ERP API, a microservice). The tool signatures stay the same; only the bodies change.

In [7]:
SHIPMENT_DB = {
    "SHP-1001": {
        "id": "SHP-1001", "origin": "Shanghai", "destination": "Long Beach",
        "status": "in_transit", "eta": "2026-05-02",
        "carrier": "Maersk", "containers": 12, "customer": "Acme Industrial",
    },
    "SHP-1002": {
        "id": "SHP-1002", "origin": "Rotterdam", "destination": "Newark",
        "status": "at_port", "eta": "2026-04-28",
        "carrier": "MSC", "containers": 6, "customer": "Belden Foods",
    },
    "SHP-1003": {
        "id": "SHP-1003", "origin": "Busan", "destination": "Oakland",
        "status": "delayed", "eta": "2026-05-10",
        "carrier": "ONE", "containers": 18, "customer": "Acme Industrial",
    },
}
print(f"Seeded {len(SHIPMENT_DB)} shipments.")

Seeded 3 shipments.


## 5b. Seed durable preferences (`preference` records)

The Assistant memory profile is *short-term context **plus long-term preference retrieval***. Preferences — how a customer wants to be notified, an operator's display habits — are exactly what the assistant should recall across sessions without being re-told.

Oracle AI Agent Memory has a first-class **`preference`** record type for this. We seed a few here; later the assistant recalls them with a dedicated tool that filters `record_types=["preference"]`, so behavioural preferences stay separate from operational state changes.

In [8]:
# Long-term PREFERENCES: how customers and the operator like to be handled.
# These are 'preference' records — a first-class type, distinct from operational 'memory'.
# Written through the store's typed add(); recalled later with a record_types filter.
PREFERENCES = [
    "Acme Industrial prefers proactive delay notifications by phone, not email.",
    "Belden Foods requires a temperature-controlled handling note on every shipment update.",
    "Operator Morgan likes shipment rundowns grouped by customer, most urgent first.",
]

pref_ids = memory_client._store.add(
    contents=PREFERENCES,
    record_type="preference",
    user_ids=OPERATOR_ID,
    agent_ids=AGENT_ID,
)
print(f"Seeded {len(pref_ids)} preferences.")

Seeded 3 preferences.


## 6. Define the agent's tools

We define five tools:

| Tool | Read/Write | Purpose |
|---|---|---|
| `list_shipments` | Read | Enumerate all known shipments |
| `get_shipment_status` | Read | Look up the current state of one shipment |
| `update_shipment` | **Write** | Mutate a field on a shipment |
| `recall_operational_notes` | Read | Semantic search over Oracle-backed `memory` records for prior context |
| `recall_customer_preferences` | Read | Semantic search over `preference` records for how a customer/operator likes to be handled |

Each is decorated with `@tool(name, description, input_schema)` and returns a structured content dict. The SDK wires the type schema, docstring, and return format into the LLM's tool-call protocol automatically.

> **💡 Key insight:** Return `{"content": [{"type": "text", "text": ...}]}` from every tool. The two recall tools return `SearchResult.formatted_content` — the prompt-ready, type-tagged rendering — and each filters on `record_types`, so operational state changes (`memory`) and behavioural preferences (`preference`) never bleed into each other.

In [9]:
import json
from claude_agent_sdk import tool, create_sdk_mcp_server


@tool(
    "list_shipments",
    "List every shipment currently tracked by the system. Returns a JSON array of shipment summaries.",
    {},
)
async def list_shipments(args: dict) -> dict:
    summaries = [
        {"id": s["id"], "status": s["status"], "eta": s["eta"], "customer": s["customer"]}
        for s in SHIPMENT_DB.values()
    ]
    return {"content": [{"type": "text", "text": json.dumps(summaries, indent=2)}]}


@tool(
    "get_shipment_status",
    "Return the full record for one shipment by its ID (e.g. 'SHP-1001').",
    {"shipment_id": str},
)
async def get_shipment_status(args: dict) -> dict:
    sid = args["shipment_id"]
    if sid not in SHIPMENT_DB:
        return {"content": [{"type": "text", "text": f"No shipment with id {sid}."}]}
    return {"content": [{"type": "text", "text": json.dumps(SHIPMENT_DB[sid], indent=2)}]}


@tool(
    "update_shipment",
    (
        "Update one field of one shipment. Use for status transitions, ETA revisions, or "
        "carrier reassignments. Writes the change through to the operational store and "
        "records the update as a durable memory so it survives restarts."
    ),
    {"shipment_id": str, "field": str, "value": str},
)
async def update_shipment(args: dict) -> dict:
    sid, field, value = args["shipment_id"], args["field"], args["value"]
    if sid not in SHIPMENT_DB:
        return {"content": [{"type": "text", "text": f"No shipment with id {sid}."}]}
    if field not in SHIPMENT_DB[sid]:
        return {"content": [{"type": "text", "text": f"Unknown field '{field}' on shipment {sid}."}]}
    before = SHIPMENT_DB[sid][field]
    SHIPMENT_DB[sid][field] = value
    # Record the change as a durable operational memory
    memory_client.add_memory(
        f"Shipment {sid} {field} changed from '{before}' to '{value}'.",
        user_id=OPERATOR_ID, agent_id=AGENT_ID,
        metadata={"shipment_id": sid, "kind": "state_change"},
    )
    return {"content": [{"type": "text",
            "text": f"Updated {sid}.{field}: '{before}' -> '{value}'."}]}


@tool(
    "recall_operational_notes",
    (
        "Semantic search over durable operational memory for prior notes relevant to the "
        "query. Use this when the operator references past events, recurring customer "
        "issues, or context that might live in memory rather than the live shipment record."
    ),
    {"query": str, "max_results": int},
)
async def recall_operational_notes(args: dict) -> dict:
    results = memory_client.search(
        args["query"], user_id=OPERATOR_ID, agent_id=AGENT_ID,
        record_types=["memory"], max_results=args.get("max_results", 5),
    )
    if not results:
        return {"content": [{"type": "text", "text": "(no prior notes)"}]}
    # formatted_content is the prompt-ready, type-tagged rendering the LLM should see.
    text = "\n".join(r.formatted_content for r in results)
    return {"content": [{"type": "text", "text": text}]}


@tool(
    "recall_customer_preferences",
    (
        "Look up durable PREFERENCES for a customer or the operator — how they like to be "
        "notified, handled, or shown information. Call this before composing a customer-facing "
        "update or deciding how to deliver a notification."
    ),
    {"query": str},
)
async def recall_customer_preferences(args: dict) -> dict:
    results = memory_client.search(
        args["query"], user_id=OPERATOR_ID, agent_id=AGENT_ID,
        record_types=["preference"], max_results=5,
    )
    if not results:
        return {"content": [{"type": "text", "text": "(no preferences on record)"}]}
    text = "\n".join(r.formatted_content for r in results)
    return {"content": [{"type": "text", "text": text}]}


print("Tools defined: list_shipments, get_shipment_status, update_shipment, "
      "recall_operational_notes, recall_customer_preferences")

Tools defined: list_shipments, get_shipment_status, update_shipment, recall_operational_notes, recall_customer_preferences


## 7. Wrap the tools in an MCP server

The Claude Agent SDK doesn't attach tools directly to the agent — it attaches MCP servers. `create_sdk_mcp_server` bundles our four tools into a server the SDK can run in-process.

> **💡 Key insight:** MCP (Model Context Protocol) is how the Claude Agent SDK surfaces tools. The SDK handles the protocol for you; `create_sdk_mcp_server` is the bridge from a list of `@tool`-decorated functions to the MCP surface the agent can see.

In [10]:
supply_server = create_sdk_mcp_server(
    name="supply-chain",
    tools=[list_shipments, get_shipment_status, update_shipment,
           recall_operational_notes, recall_customer_preferences],
)
print("MCP server created: supply-chain")

MCP server created: supply-chain


## 8. Configure the assistant with options

`ClaudeAgentOptions` holds the system prompt, the MCP servers the agent can see, and the tool allowlist. Tool names on the allowlist use the `mcp__<server>__<tool>` convention.

> **📌 Permission mode.** `acceptEdits` auto-approves tool calls — appropriate for a single-operator assistant where the tools are safe. In production you'd likely use `default` and surface tool-use prompts to the operator before mutating state.

In [11]:
from claude_agent_sdk import ClaudeAgentOptions

SYSTEM_PROMPT = """You are a supply chain assistant for the West Coast logistics desk.

Operating rules:
1. When the operator asks about a shipment, use `get_shipment_status` or `list_shipments` first
   rather than speculating from context.
2. Before changing state, confirm the intended change in plain English, then call `update_shipment`.
3. When the operator mentions a customer, carrier, or prior event, call `recall_operational_notes`
   to check for relevant prior context.
4. Before composing a customer-facing update or deciding how to notify someone, call
   `recall_customer_preferences` and honour what it returns (e.g. preferred channel).
5. Be concise. Surface concrete shipment IDs, fields, and values. Avoid restating what the tool
   returned — interpret it for the operator.
"""

options = ClaudeAgentOptions(
    system_prompt=SYSTEM_PROMPT,
    mcp_servers={"sc": supply_server},
    allowed_tools=[
        "mcp__sc__list_shipments",
        "mcp__sc__get_shipment_status",
        "mcp__sc__update_shipment",
        "mcp__sc__recall_operational_notes",
        "mcp__sc__recall_customer_preferences",
    ],
    permission_mode="acceptEdits",
)
print("Agent options configured.")

Agent options configured.


## 9. Run a multi-turn conversation with `ClaudeSDKClient`

`ClaudeSDKClient` is the multi-turn interface — unlike `query()`, it maintains context across calls. We'll use it as an async context manager and feed a scripted conversation through it, simulating an operator working through their morning shipments.

We also write each turn to an Oracle memory thread so we have a durable log of the conversation for audit, training, or continuity purposes.

> **💡 Key insight:** Use `query()` for one-shot tasks, `ClaudeSDKClient` for ongoing conversations. The client's context window grows with each turn until you close it — that's what makes multi-turn coherence work.

In [12]:
from claude_agent_sdk import ClaudeSDKClient, AssistantMessage, ToolUseBlock, TextBlock
from oracleagentmemory.apis.thread import Message

# Open a thread to log the conversation into OAMP
conv_thread = memory_client.create_thread(user_id=OPERATOR_ID, agent_id=AGENT_ID)

operator_turns = [
    "What shipments are in flight right now?",
    "What's going on with SHP-1003? The customer just asked for an update.",
    "The carrier says SHP-1003 will now arrive on August 14 because of port congestion. Update the ETA.",
    "Remind me who the customer is on SHP-1003, and flag anything you know about them.",
    "Draft a short delay update for Acme on SHP-1003, and tell me how they want it delivered.",
]

async with ClaudeSDKClient(options=options) as client:
    for turn_idx, prompt in enumerate(operator_turns, 1):
        print(f"\n{'=' * 70}\nTurn {turn_idx} — OPERATOR: {prompt}\n{'=' * 70}")
        await client.query(prompt)
        # Log operator turn
        conv_thread.add_messages([Message(role="user", content=prompt)])

        assistant_text_parts = []
        async for message in client.receive_response():
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, TextBlock):
                        print(f"ASSISTANT: {block.text}")
                        assistant_text_parts.append(block.text)
                    elif isinstance(block, ToolUseBlock):
                        print(f"  [tool] {block.name}({block.input})")
        # Log assistant turn
        if assistant_text_parts:
            conv_thread.add_messages([
                Message(role="assistant", content="\n".join(assistant_text_parts)),
            ])

warn: CPU lacks AVX support, strange crashes may occur. Reinstall Bun or use *-baseline build:
  https://github.com/oven-sh/bun/releases/download/bun-v1.3.14/bun-darwin-x64-baseline.zip



Turn 1 — OPERATOR: What shipments are in flight right now?


oracleagentmemory/core/thread.py:463: UserWarning: You are calling an asynchronous method in a synchronous method from an asynchronous context. This is highly discouraged because it can lead to deadlocks. Please use the asynchronous method equivalent: add_messages_async


ASSISTANT: I'll pull the current shipment list. Let me load the tool first.
  [tool] ToolSearch({'query': 'select:mcp__sc__list_shipments', 'max_results': 1})
  [tool] mcp__sc__list_shipments({})
ASSISTANT: Three active shipments, none delivered yet:

- **SHP-1001** — Acme Industrial — `in_transit`, ETA 2026-05-02 (the one truly moving)
- **SHP-1002** — Belden Foods — `at_port`, ETA 2026-04-28 (sitting at port, not yet moving inland)
- **SHP-1003** — Acme Industrial — `delayed`, ETA 2026-05-10 (behind schedule — worth a look)

Heads-up: all three ETAs are in the past relative to today (2026-06-16), so the displayed dates are stale. Want me to pull full status on any of these — SHP-1003's delay seems the most pressing?

Turn 2 — OPERATOR: What's going on with SHP-1003? The customer just asked for an update.
ASSISTANT: Let me pull the full status, plus prior context and the customer's update preferences.
  [tool] ToolSearch({'query': 'select:mcp__sc__get_shipment_status,mcp__sc__recall_o

oracleagentmemory/core/oracleagentmemory.py:853: UserWarning: You are calling an asynchronous method in a synchronous method from an asynchronous context. This is highly discouraged because it can lead to deadlocks. Please use the asynchronous method equivalent: 


ASSISTANT: Here's the picture on **SHP-1003**:

- **Lane:** Busan → Oakland, carrier **ONE**, 18 containers
- **Status:** `delayed`. System record still shows ETA **2026-05-10** (now past due)
- **Cause:** port congestion. Operational notes carry a **revised ETA of May 14**, but that date is flagged as *not yet firm* — there's a note saying any update should go out **only once ONE confirms a firm ETA**.
- **Customer ask:** Acme has an outstanding urgent request for a status update on this shipment.

**How to reach them:** Acme Industrial has a standing preference for **proactive phone calls, not email**, on delay notifications.

One caveat worth flagging: both the record ETA (May 10) and the revised note ETA (May 14) are already in the past relative to today (June 16), so the dates look stale — worth re-confirming with ONE before you call.

Recommendation: get a firm ETA from ONE first, then phone Acme. Want me to draft the phone-call talking points, or update the shipment record once 

## 10. Inspect the memory state

Two things landed in Oracle during the conversation:

1. **Operational memories** written by `update_shipment` (state changes) and by the automatic extractor (from the conversation thread).
2. **Conversation messages** on the thread, which also feed the extractor to create further durable memories.

Let's look at both.

In [13]:
# Durable operational memories relevant to shipment 1003 (record_types filter keeps preferences out)
hits = memory_client.search(
    "SHP-1003 ETA change", user_id=OPERATOR_ID, agent_id=AGENT_ID,
    record_types=["memory"], max_results=10,
)
print(f"Durable memories related to SHP-1003: {len(hits)}")
for r in hits:
    print(f"  - {r.content}")

# Raw thread log (short-term conversation history)
messages = conv_thread.get_messages()
print(f"\nConversation messages on thread: {len(messages)}")
for m in messages[:4]:
    print(f"  [{m.role}] {m.content[:100]}...")

# Confirm the live shipment record was mutated too
print(f"\nCurrent SHP-1003 ETA: {SHIPMENT_DB['SHP-1003']['eta']}")

Durable memories related to SHP-1003: 10
  - Shipment SHP-1003 eta changed from '2026-05-10' to '2026-08-14'.
  - SHP-1003 (Acme Industrial, Busan→Oakland, carrier ONE, 18 containers): proposed ETA update to 2026-05-14 is already ~5 weeks in the past as of 2026-06-16. Current record shows ETA 2026-05-10. Awaiting user confirmation that 2026-05-14 is correct before updating.
  - SHP-1003 ETA update from carrier ONE is ambiguous: May 14, 2026 is already past (current date 2026-06-16); clarification needed on whether intended ETA is June 14, 2026 or if May 14 is correct before updating record.
  - Customer requested specific update on SHP-1003 (delayed shipment, ETA 2026-05-10, overdue as of 2026-06-16).
  - All three active shipments (SHP-1001, SHP-1002, SHP-1003) have past ETAs as of 2026-06-16 and require active follow-up.
  - All three active shipments (SHP-1001, SHP-1002, SHP-1003) have ETAs in the past; investigation required.
  - SHP-1003 (Acme, Busan→Oakland, ONE carrier): ETA on 

## 10b. Consolidate working memory — running summary and context card

Across a long operator session the raw turn-by-turn history grows unbounded. Oracle AI Agent Memory consolidates it for you, directly on the thread:

- **`thread.get_summary()`** → an LLM-written summary of the session so far.
- **`thread.get_context_card()`** → a compact, prompt-ready *context card* — the working-memory block you'd inject at the top of the next turn instead of replaying every message.

> **💡 Key insight:** This is the short-term tier of the Assistant memory profile. Durable `memory`/`preference` records are the long-term tier (recalled by search); the summary and context card are the *working* tier — they keep a long session inside the context window without losing the thread.

In [14]:
# Working memory: a running summary and a prompt-ready context card of the live session,
# both produced by the thread's LLM from the messages we logged during the conversation.
summary = conv_thread.get_summary()
print("=== Session summary (get_summary) ===")
print(summary.content)

card = conv_thread.get_context_card()
print("\n=== Context card (get_context_card.formatted_content) ===")
print(card.formatted_content[:1000])

oracleagentmemory/core/thread.py:792: UserWarning: You are calling an asynchronous method in a synchronous method from an asynchronous context. This is highly discouraged because it can lead to deadlocks. Please use the asynchronous method equivalent: get_summary_async


=== Session summary (get_summary) ===
The user asked for active shipments and received a list of three (SHP-1001, SHP-1002, SHP-1003), all with stale ETAs. Focus shifted to SHP-1003 (Acme Industrial, Busan→Oakland, carrier ONE, delayed by port congestion), whose ETA was updated from 2026-05-10 to 2026-08-14 after carrier confirmation. The assistant drafted phone call talking points for Acme, noting their preference for proactive phone calls over email, and flagged that SHP-1001 is also overdue for the same customer.


oracleagentmemory/core/thread.py:707: UserWarning: You are calling an asynchronous method in a synchronous method from an asynchronous context. This is highly discouraged because it can lead to deadlocks. Please use the asynchronous method equivalent: get_context_card_async



=== Context card (get_context_card.formatted_content) ===
<context_card>
  <topics>
    <topic>carrier one busan oakland</topic>
    <topic>shp-1001 acme industrial overdue</topic>
    <topic>active shipments in transit</topic>
    <topic>shp-1003 delay port congestion</topic>
    <topic>shipment status management</topic>
    <topic>delay notification phone call script</topic>
    <topic>acme industrial customer profile</topic>
    <topic>eta update 2026-08-14</topic>
  </topics>
  <summary>
    The user is managing active shipments and investigating a delay on SHP-1003 (Acme Industrial, Busan→Oakland, carrier ONE, 18 containers). The ETA was updated from 2026-05-10 to 2026-08-14 due to port congestion after receiving a firm date from the carrier. A phone call script was drafted for Acme Industrial, who prefers proactive phone calls over email for delay notifications.
  </summary>
  <relevant_information>
    <memory>
      <timestamp>2026-06-16</timestamp>
      <content>SHP-1003 (Ac

## 11. Verify continuity — operator returns later

A later session should pick up the operational context without the operator re-explaining. We create a fresh client and ask a question that relies on memory from the earlier turn.

> **💡 Why this works — search scope.** The recall tools call `memory_client.search(...)`, which defaults to `exact_thread_match=False`. That means relevant `memory` and `preference` records from *earlier threads* are in scope as long as the `user_id`/`agent_id` match. Continuity across sessions is a property of the search scope, not a manual replay. (Set `exact_thread_match=True` when you deliberately want to confine recall to the current thread.)

In [15]:
followup = "Earlier we revised an ETA for a shipment headed to Oakland. What's the revised ETA and why did it change?"

print(f"OPERATOR (new session): {followup}\n")

async with ClaudeSDKClient(options=options) as client2:
    await client2.query(followup)
    async for message in client2.receive_response():
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"ASSISTANT: {block.text}")
                elif isinstance(block, ToolUseBlock):
                    print(f"  [tool] {block.name}({block.input})")

OPERATOR (new session): Earlier we revised an ETA for a shipment headed to Oakland. What's the revised ETA and why did it change?



warn: CPU lacks AVX support, strange crashes may occur. Reinstall Bun or use *-baseline build:
  https://github.com/oven-sh/bun/releases/download/bun-v1.3.14/bun-darwin-x64-baseline.zip


ASSISTANT: I'll look up the Oakland shipment and check for prior context on why the ETA changed. Let me first load the tools I need.
  [tool] ToolSearch({'query': 'select:mcp__sc__list_shipments,mcp__sc__get_shipment_status,mcp__sc__recall_operational_notes', 'max_results': 5})
  [tool] mcp__sc__list_shipments({})
ASSISTANT: The list doesn't show destinations. Let me pull the full records and check operational memory for the Oakland ETA revision.
  [tool] mcp__sc__get_shipment_status({'shipment_id': 'SHP-1001'})
  [tool] mcp__sc__get_shipment_status({'shipment_id': 'SHP-1002'})
  [tool] mcp__sc__get_shipment_status({'shipment_id': 'SHP-1003'})
  [tool] mcp__sc__recall_operational_notes({'query': 'Oakland shipment revised ETA reason for change', 'max_results': 5})
ASSISTANT: The Oakland shipment is **SHP-1003** (Acme Industrial, Busan → Oakland, carrier ONE, 18 containers).

- **Revised ETA: 2026-08-14** (status: `delayed`). The record was changed from **2026-05-10 → 2026-08-14**.
- **W

## 12. Cleanup

In [16]:
# Uncomment to delete the conversation thread and its messages
# memory_client.delete_thread(conv_thread.thread_id)

connection.close()
print("Connection closed.")

Connection closed.


## Key Takeaways

> **🎯 1. `@tool` + `create_sdk_mcp_server` is the Claude Agent SDK's tool-definition pattern.** You don't pass bare Python functions to the agent — you wrap them in an MCP server and point the agent's options at it. The indirection pays off when you scale to multiple tool bundles.

> **🎯 2. Tools should return MCP content blocks — and return `formatted_content`.** `{"content": [{"type": "text", "text": ...}]}` is the canonical shape. For recall tools, put `SearchResult.formatted_content` in the text: it's the prompt-ready, type-tagged rendering, so the model sees what kind of record it's reading.

> **🎯 3. Assistants need three memory tiers, and they map to three record kinds.** *Working* tier = the live thread (`get_summary` / `get_context_card`). *Long-term operational* tier = `memory` records (what changed and why). *Long-term preference* tier = `preference` records (how a customer/operator wants to be handled). Filtering on `record_types` keeps them from bleeding into each other.

> **🎯 4. Every write that matters should hit memory, not just the live store.** The `update_shipment` tool mutates the live shipment record *and* writes a memory record describing the change. The first is the system of record; the second is how the assistant stays coherent with the operator about what changed and why.

> **🎯 5. Continuity is a search-scope property.** A fresh session recalls prior context because `search()` defaults to `exact_thread_match=False` over the same `user_id`/`agent_id`. The live shipment store, the durable memories/preferences, the conversation thread, and its summary can all live in one Oracle database — one pool, one backup, one compliance review.